In [ ]:
# Dependency setup. This uses Python subprocess instead of notebook shell magic so it works in Kaggle and Colab.
import importlib.util, subprocess, sys

required = {
    'timm': 'timm>=0.9.16',
    'transformers': 'transformers>=4.40.0',
    'sklearn': 'scikit-learn',
    'cv2': 'opencv-python-headless',
    'tqdm': 'tqdm',
    'PIL': 'pillow',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('[setup] installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('[setup] base packages already available')

# Optional but important for the stronger run. If this fails, the notebook falls back to global-only graph clustering.
if importlib.util.find_spec('lightglue') is None:
    print('[setup] trying to install LightGlue for local verification')
    code = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/cvg/LightGlue'])
    print('[setup] LightGlue install exit code:', code)
else:
    print('[setup] LightGlue already available')


In [ ]:
# Experiment configuration. Keep this cell as the main control panel.
VERSION = 'wildfusion_graph_v20260423'
SEED = 20260423

# L4 defaults. If you hit OOM, reduce these; the extractor retries smaller batches automatically.
BATCH_SIZE_MEGA = 48
BATCH_SIZE_MIEW = 64
NUM_WORKERS = 2

# Graph construction.
TEST_TEST_TOPK = {
    'LynxID2025': 40,
    'SalamanderID2025': 40,
    'SeaTurtleID2022': 35,
    'TexasHornedLizards': 45,
}

# Local matching is the main expensive upgrade. It is applied only to shortlisted test-test graph edges.
RUN_LOCAL_MATCHING = True
LOCAL_PREFILTER_BELOW_THRESHOLD = 0.08
LOCAL_IMAGE_SIZE = 512
LOCAL_KEYPOINTS = 1024
LOCAL_PAIR_CAP = {
    'LynxID2025': 6000,
    'SalamanderID2025': 5000,
    'SeaTurtleID2022': 3500,
    'TexasHornedLizards': 3000,
}

# Conservative fallbacks used only if validation cannot tune a species.
FALLBACK_THRESHOLDS = {
    'LynxID2025': {'known_thr': 0.64, 'edge_thr': 0.68, 'margin_thr': 0.02},
    'SalamanderID2025': {'known_thr': 0.68, 'edge_thr': 0.72, 'margin_thr': 0.02},
    'SeaTurtleID2022': {'known_thr': 0.66, 'edge_thr': 0.70, 'margin_thr': 0.02},
    'TexasHornedLizards': {'known_thr': 1.10, 'edge_thr': 0.70, 'margin_thr': 0.00},
}

print('version:', VERSION)


In [ ]:
from pathlib import Path
import gc, json, math, os, random, time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.metrics import adjusted_rand_score

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

ImageFile.LOAD_TRUNCATED_IMAGES = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = WORK_DIR / 'animalclef_wildfusion_graph'
CACHE_DIR = OUT_DIR / 'cache'
SUB_DIR = OUT_DIR / 'submissions'
REPORT_DIR = OUT_DIR / 'reports'
for d in [OUT_DIR, CACHE_DIR, SUB_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
    try:
        _ = torch.zeros(1, device='cuda') + 1
        print('[cuda] smoke test passed')
    except Exception as e:
        raise RuntimeError('CUDA is visible but unusable. Switch runtime/GPU or reinstall compatible torch.') from e

def find_competition_root():
    candidates = [
        Path('/kaggle/input/animal-clef-2026'),
        Path('/kaggle/input/competitions/animal-clef-2026'),
        Path('/content/animal-clef-2026'),
        Path.cwd() / 'animal-clef-2026',
    ]
    for root in candidates:
        if (root / 'metadata.csv').exists() and (root / 'sample_submission.csv').exists():
            return root
    search_roots = [Path('/kaggle/input'), Path('/content'), Path.cwd()]
    for base in search_roots:
        if not base.exists():
            continue
        for meta_path in base.rglob('metadata.csv'):
            root = meta_path.parent
            if (root / 'sample_submission.csv').exists():
                return root
    raise FileNotFoundError('Could not find AnimalCLEF2026 metadata.csv and sample_submission.csv')

DATA_ROOT = find_competition_root()
print('DATA_ROOT:', DATA_ROOT)

meta = pd.read_csv(DATA_ROOT / 'metadata.csv')
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
meta = meta.reset_index(drop=True)
meta['row_idx'] = np.arange(len(meta))
meta['abs_path'] = meta['path'].map(lambda p: str((DATA_ROOT / str(p)).resolve()) if not Path(str(p)).is_absolute() else str(p))
meta['date_dt'] = pd.to_datetime(meta.get('date'), errors='coerce')
meta['orientation'] = meta.get('orientation', '').fillna('').astype(str)

print(meta.groupby(['dataset', 'split']).size())
print('sample rows:', len(sample))
assert set(sample['image_id']).issubset(set(meta['image_id'])), 'sample_submission has image_id not present in metadata'

def l2_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

class PathImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (512, 512), (0, 0, 0))
        return self.transform(img)

def output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        x = out
    elif isinstance(out, dict):
        for key in ['embedding', 'embeddings', 'features', 'pooler_output', 'last_hidden_state']:
            if key in out and out[key] is not None:
                x = out[key]
                break
        else:
            raise TypeError(f'Unsupported dict model output keys: {list(out.keys())}')
    elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
        x = out.pooler_output
    elif hasattr(out, 'last_hidden_state'):
        x = out.last_hidden_state.mean(dim=1)
    elif isinstance(out, (tuple, list)):
        x = out[0]
    else:
        raise TypeError(f'Unsupported model output type: {type(out)}')
    if x.ndim == 4:
        x = torch.nn.functional.adaptive_avg_pool2d(x, 1).flatten(1)
    if x.ndim == 3:
        x = x.mean(dim=1)
    return x

def extract_features(model, paths, transform, cache_path, batch_size):
    cache_path = Path(cache_path)
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.shape[0] == len(paths):
            print('[cache] loaded', cache_path, arr.shape)
            return l2_normalize(arr)
        print('[cache] ignoring mismatched cache', cache_path, arr.shape)
    ds = PathImageDataset(paths, transform)
    bs = int(batch_size)
    while bs >= 1:
        feats = []
        try:
            loader = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))
            model.eval()
            with torch.inference_mode():
                for batch in tqdm(loader, desc=f'extract bs={bs}'):
                    batch = batch.to(DEVICE, non_blocking=True)
                    if DEVICE == 'cuda':
                        with torch.autocast(device_type='cuda', dtype=torch.float16):
                            out = model(batch)
                    else:
                        out = model(batch)
                    feat = output_to_tensor(out).detach().float().cpu().numpy()
                    feats.append(feat)
            arr = l2_normalize(np.concatenate(feats, axis=0))
            np.save(cache_path, arr.astype(np.float16))
            print('[cache] wrote', cache_path, arr.shape)
            return arr
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower() or bs == 1:
                raise
            print('[oom] reducing batch size', bs, '->', bs // 2)
            bs = max(1, bs // 2)
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

def load_megadescriptor():
    import timm
    name = 'hf-hub:BVRA/MegaDescriptor-L-384'
    try:
        model = timm.create_model(name, pretrained=True, num_classes=0)
    except TypeError:
        model = timm.create_model(name, pretrained=True)
    return model.to(DEVICE)

def load_miew():
    from transformers import AutoModel
    # MiewID remote code is slightly behind newer Transformers loaders. Newer Transformers
    # calls mark_tied_weights_as_initialized(), which expects this attribute to exist.
    from transformers.modeling_utils import PreTrainedModel
    if not hasattr(PreTrainedModel, 'all_tied_weights_keys'):
        PreTrainedModel.all_tied_weights_keys = {}
    model = AutoModel.from_pretrained('conservationxlabs/miewid-msv3', trust_remote_code=True)
    if not hasattr(model, 'all_tied_weights_keys'):
        model.all_tied_weights_keys = {}
    return model.to(DEVICE)

all_paths = meta['abs_path'].tolist()
mega_transform = T.Compose([
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
miew_transform = T.Compose([
    T.Resize((440, 440)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('[model] MegaDescriptor-L-384')
mega_model = load_megadescriptor()
mega = extract_features(mega_model, all_paths, mega_transform, CACHE_DIR / 'mega_l384_all.npy', BATCH_SIZE_MEGA)
del mega_model
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

print('[model] MiewID-msv3')
miew_model = load_miew()
miew = extract_features(miew_model, all_paths, miew_transform, CACHE_DIR / 'miew_msv3_all.npy', BATCH_SIZE_MIEW)
del miew_model
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

features = l2_normalize(np.concatenate([0.62 * mega, 0.38 * miew], axis=1))
print('features:', features.shape)


In [ ]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.sz = [1] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.sz[ra] < self.sz[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        self.sz[ra] += self.sz[rb]

def side_bucket(orientation):
    o = str(orientation).lower()
    if 'left' in o:
        return 'left'
    if 'right' in o:
        return 'right'
    return None

def apply_orientation_penalty(sim, q_meta, g_meta, species):
    if species not in {'LynxID2025', 'SeaTurtleID2022'}:
        return sim
    q_side = [side_bucket(x) for x in q_meta['orientation'].tolist()]
    g_side = np.array([side_bucket(x) for x in g_meta['orientation'].tolist()], dtype=object)
    penalty = 0.075 if species == 'LynxID2025' else 0.055
    for i, s in enumerate(q_side):
        if s is None:
            continue
        mask = np.array([(t is not None and t != s) for t in g_side])
        if mask.any():
            sim[i, mask] -= penalty
    return sim

def build_open_validation_indices(sp_df, species):
    sp_df = sp_df[sp_df['split'].eq('train')].copy()
    usable_dates = sp_df['date_dt'].notna().mean() > 0.5
    if usable_dates:
        cutoff = sp_df['date_dt'].quantile(0.80)
        gallery = sp_df[sp_df['date_dt'] < cutoff]
        query = sp_df[sp_df['date_dt'] >= cutoff]
        if len(gallery) >= 50 and len(query) >= 30 and query['identity'].nunique() >= 5:
            return gallery['row_idx'].to_numpy(), query['row_idx'].to_numpy(), 'time80_open'
    species_seed = SEED + sum((i + 1) * ord(ch) for i, ch in enumerate(species))
    rng = np.random.default_rng(species_seed)
    gallery_idx, query_idx = [], []
    ids = np.array(sorted(sp_df['identity'].dropna().unique()))
    rng.shuffle(ids)
    novel_ids = set(ids[:max(1, int(0.22 * len(ids)))])
    for ident, group in sp_df.groupby('identity'):
        rows = group['row_idx'].to_numpy()
        rng.shuffle(rows)
        if ident in novel_ids:
            query_idx.extend(rows.tolist())
        elif len(rows) >= 2:
            query_idx.append(int(rows[0]))
            gallery_idx.extend(rows[1:].tolist())
        else:
            gallery_idx.extend(rows.tolist())
    return np.array(gallery_idx, dtype=int), np.array(query_idx, dtype=int), 'identity_open_random'

def prepare_graph(query_idx, gallery_idx, species, topk):
    q_meta = meta.loc[query_idx].reset_index(drop=True)
    q_feat = features[query_idx]
    if len(gallery_idx):
        g_meta = meta.loc[gallery_idx].reset_index(drop=True)
        sim_qg = q_feat @ features[gallery_idx].T
        sim_qg = apply_orientation_penalty(sim_qg, q_meta, g_meta, species)
        gallery_labels = g_meta['identity'].astype(str).to_numpy()
    else:
        sim_qg = None
        gallery_labels = np.array([])
    sim_qq = q_feat @ q_feat.T
    sim_qq = apply_orientation_penalty(sim_qq, q_meta, q_meta, species)
    np.fill_diagonal(sim_qq, -9.0)
    k = min(int(topk), max(1, len(query_idx) - 1))
    nbr = np.argpartition(-sim_qq, kth=np.arange(k), axis=1)[:, :k]
    nbr_sets = [set(row.tolist()) for row in nbr]
    edges = []
    for i in range(len(query_idx)):
        for j in nbr[i]:
            j = int(j)
            if j <= i:
                continue
            if i in nbr_sets[j]:
                edges.append((i, j, float(sim_qq[i, j])))
    edges.sort(key=lambda x: x[2], reverse=True)
    return {
        'species': species,
        'query_idx': np.asarray(query_idx, dtype=int),
        'gallery_idx': np.asarray(gallery_idx, dtype=int),
        'q_meta': q_meta,
        'sim_qg': sim_qg,
        'gallery_labels': gallery_labels,
        'edges': edges,
    }

def predict_from_prepared(prep, known_thr, edge_thr, margin_thr):
    n = len(prep['query_idx'])
    dsu = DSU(n)
    assigned = np.array([None] * n, dtype=object)
    if prep['sim_qg'] is not None and prep['sim_qg'].shape[1] > 0:
        sim = prep['sim_qg']
        order = np.argpartition(-sim, kth=np.arange(min(2, sim.shape[1])), axis=1)[:, :min(2, sim.shape[1])]
        for i in range(n):
            cand = order[i]
            cand = cand[np.argsort(-sim[i, cand])]
            top = int(cand[0])
            top_score = float(sim[i, top])
            second = float(sim[i, cand[1]]) if len(cand) > 1 else -1.0
            if top_score >= known_thr and (top_score - second) >= margin_thr:
                assigned[i] = prep['gallery_labels'][top]
        groups = defaultdict(list)
        for i, lab in enumerate(assigned):
            if lab is not None:
                groups[lab].append(i)
        for members in groups.values():
            for m in members[1:]:
                dsu.union(members[0], m)
    for i, j, score in prep['edges']:
        if score < edge_thr:
            break
        ai, aj = assigned[i], assigned[j]
        if ai is not None and aj is not None and ai != aj:
            continue
        if (ai is None) ^ (aj is None):
            if score < edge_thr + 0.035:
                continue
        dsu.union(i, j)
    root_to_known = defaultdict(list)
    for i, lab in enumerate(assigned):
        if lab is not None:
            root_to_known[dsu.find(i)].append(lab)
    root_name = {}
    novel_counter = 0
    pred = []
    for i in range(n):
        r = dsu.find(i)
        if r not in root_name:
            if root_to_known.get(r):
                root_name[r] = Counter(root_to_known[r]).most_common(1)[0][0]
            else:
                root_name[r] = f'novel_{novel_counter}'
                novel_counter += 1
        pred.append(root_name[r])
    return np.array(pred, dtype=object)

def tune_species(species):
    sp_df = meta[meta['dataset'].eq(species)]
    if species == 'TexasHornedLizards' or sp_df[sp_df['split'].eq('train')].empty:
        return FALLBACK_THRESHOLDS[species] | {'local_ari': 'N/A', 'split': 'no_train'}
    gallery_idx, query_idx, split_name = build_open_validation_indices(sp_df, species)
    prep = prepare_graph(query_idx, gallery_idx, species, TEST_TEST_TOPK.get(species, 40))
    true = meta.loc[query_idx, 'identity'].astype(str).to_numpy()
    if len(np.unique(true)) < 2:
        return FALLBACK_THRESHOLDS[species] | {'local_ari': 'N/A', 'split': split_name}
    known_grid = np.round(np.linspace(0.50, 0.86, 10), 3)
    edge_grid = np.round(np.linspace(0.54, 0.88, 12), 3)
    margin_grid = [0.00, 0.015, 0.03, 0.05]
    best = {'local_ari': -99, 'known_thr': None, 'edge_thr': None, 'margin_thr': None, 'split': split_name}
    for kt in known_grid:
        for et in edge_grid:
            for mt in margin_grid:
                pred = predict_from_prepared(prep, float(kt), float(et), float(mt))
                ari = adjusted_rand_score(true, pred)
                if ari > best['local_ari']:
                    best.update({'local_ari': float(ari), 'known_thr': float(kt), 'edge_thr': float(et), 'margin_thr': float(mt)})
    return best

thresholds = {}
for species in sorted(meta['dataset'].unique()):
    print('\n[tune]', species)
    thresholds[species] = tune_species(species)
    print(thresholds[species])

with open(REPORT_DIR / f'thresholds_{VERSION}.json', 'w') as f:
    json.dump(thresholds, f, indent=2)

valid_aris = [v['local_ari'] for v in thresholds.values() if isinstance(v.get('local_ari'), float)]
overall_local_ari = float(np.mean(valid_aris)) if valid_aris else None
print('mean local validation ARI over tunable species:', overall_local_ari)


In [ ]:
def init_local_matcher():
    if not RUN_LOCAL_MATCHING or DEVICE != 'cuda':
        return None
    try:
        from lightglue import ALIKED, LightGlue
        from lightglue.utils import load_image, rbd
        extractor = ALIKED(max_num_keypoints=LOCAL_KEYPOINTS).eval().to(DEVICE)
        matcher = LightGlue(features='aliked').eval().to(DEVICE)
        return {'kind': 'lightglue_aliked', 'extractor': extractor, 'matcher': matcher, 'load_image': load_image, 'rbd': rbd}
    except Exception as e:
        print('[local] LightGlue unavailable, local matching disabled:', repr(e))
        return None

LOCAL = init_local_matcher()

def lightglue_pair_score(local, path0, path1):
    import cv2
    load_image = local['load_image']
    rbd = local['rbd']
    extractor = local['extractor']
    matcher = local['matcher']
    with torch.inference_mode():
        image0 = load_image(str(path0), resize=LOCAL_IMAGE_SIZE).to(DEVICE)
        image1 = load_image(str(path1), resize=LOCAL_IMAGE_SIZE).to(DEVICE)
        feats0 = extractor.extract(image0)
        feats1 = extractor.extract(image1)
        matches01 = matcher({'image0': feats0, 'image1': feats1})
        feats0, feats1, matches01 = [rbd(x) for x in [feats0, feats1, matches01]]
        matches = matches01.get('matches', None)
        if matches is None or len(matches) == 0:
            return 0.0
        matches_np = matches.detach().cpu().numpy()
        n_match = len(matches_np)
        score_match = min(1.0, n_match / 120.0)
        inlier_ratio = 0.0
        if n_match >= 8:
            k0 = feats0['keypoints'][matches[:, 0]].detach().cpu().numpy()
            k1 = feats1['keypoints'][matches[:, 1]].detach().cpu().numpy()
            try:
                _, mask = cv2.findHomography(k0, k1, cv2.RANSAC, 5.0)
                if mask is not None:
                    inlier_ratio = float(mask.mean())
            except Exception:
                inlier_ratio = 0.0
        return 0.65 * score_match + 0.35 * min(1.0, inlier_ratio)

def apply_local_rerank(prep, species, edge_thr):
    if LOCAL is None:
        return prep, {'enabled': False, 'pairs': 0}
    cap = int(LOCAL_PAIR_CAP.get(species, 3000))
    min_score = edge_thr - LOCAL_PREFILTER_BELOW_THRESHOLD
    candidates = [e for e in prep['edges'] if e[2] >= min_score]
    # Prioritize edge decisions around the threshold; those are the pairs that change ARI most.
    candidates.sort(key=lambda e: abs(e[2] - edge_thr))
    candidates = candidates[:cap]
    q_meta = prep['q_meta']
    local_scores = {}
    for i, j, global_score in tqdm(candidates, desc=f'local {species}'):
        p0 = q_meta.loc[i, 'abs_path']
        p1 = q_meta.loc[j, 'abs_path']
        try:
            ls = lightglue_pair_score(LOCAL, p0, p1)
        except Exception:
            ls = None
        if ls is not None:
            local_scores[(i, j)] = float(ls)
    new_edges = []
    for i, j, global_score in prep['edges']:
        ls = local_scores.get((i, j))
        if ls is None:
            score = global_score
        else:
            score = 0.82 * global_score + 0.18 * ls
            if ls > 0.55 and global_score > edge_thr - 0.10:
                score = max(score, edge_thr + 0.01)
            if ls < 0.08 and global_score < edge_thr + 0.025:
                score = min(score, edge_thr - 0.01)
        new_edges.append((i, j, float(score)))
    new_edges.sort(key=lambda x: x[2], reverse=True)
    prep = dict(prep)
    prep['edges'] = new_edges
    return prep, {'enabled': True, 'pairs': len(local_scores), 'matcher': LOCAL['kind']}

submission_parts = []
run_notes = {'version': VERSION, 'thresholds': thresholds, 'local_matching': {}}

for species in sorted(meta['dataset'].unique()):
    sp = meta[meta['dataset'].eq(species)]
    query_idx = sp[sp['split'].eq('test')]['row_idx'].to_numpy()
    gallery_idx = sp[sp['split'].eq('train')]['row_idx'].to_numpy()
    topk = TEST_TEST_TOPK.get(species, 40)
    prep = prepare_graph(query_idx, gallery_idx, species, topk)
    t = thresholds.get(species, FALLBACK_THRESHOLDS[species])
    known_thr = float(t['known_thr'])
    edge_thr = float(t['edge_thr'])
    margin_thr = float(t['margin_thr'])
    print('\n[predict]', species, 'n_test=', len(query_idx), 'thresholds=', known_thr, edge_thr, margin_thr)
    prep, local_info = apply_local_rerank(prep, species, edge_thr)
    run_notes['local_matching'][species] = local_info
    pred = predict_from_prepared(prep, known_thr, edge_thr, margin_thr)
    q_meta = prep['q_meta'].copy()
    label_map = {}
    out_labels = []
    next_id = 0
    for lab in pred:
        lab = str(lab)
        if lab not in label_map:
            safe = ''.join(ch if ch.isalnum() or ch in ['_', '-'] else '_' for ch in lab)
            label_map[lab] = f'cluster_{species}_{safe}' if not lab.startswith('novel_') else f'cluster_{species}_{next_id}'
            if lab.startswith('novel_'):
                next_id += 1
        out_labels.append(label_map[lab])
    part = pd.DataFrame({'image_id': q_meta['image_id'].to_numpy(), 'cluster': out_labels})
    print(part['cluster'].nunique(), 'clusters; singleton-ish labels sample:', part['cluster'].head().tolist())
    submission_parts.append(part)

sub = pd.concat(submission_parts, ignore_index=True)
sub = sample[['image_id']].merge(sub, on='image_id', how='left')
assert sub['cluster'].notna().all(), 'missing predictions for some sample rows'
assert len(sub) == len(sample), 'submission row count mismatch'
assert list(sub.columns) == ['image_id', 'cluster']

versioned_path = SUB_DIR / f'submission_{VERSION}.csv'
sub.to_csv(versioned_path, index=False)
sub.to_csv(WORK_DIR / 'submission.csv', index=False)
print('wrote:', versioned_path)
print('wrote:', WORK_DIR / 'submission.csv')
print(sub.head())

run_notes['mean_local_validation_ari'] = overall_local_ari
run_notes['submission'] = str(versioned_path)
run_notes['n_rows'] = int(len(sub))
run_notes['n_clusters_by_species'] = sub.assign(dataset=sub['cluster'].str.extract(r'^cluster_([^_]+(?:_[^_]+)?)_', expand=False)).groupby(sub['cluster'].str.extract(r'^cluster_(LynxID2025|SalamanderID2025|SeaTurtleID2022|TexasHornedLizards)_', expand=False))['cluster'].nunique().to_dict()
with open(REPORT_DIR / f'run_summary_{VERSION}.json', 'w') as f:
    json.dump(run_notes, f, indent=2)

html = f"""
<html><head><title>{VERSION} run summary</title></head><body>
<h1>AnimalCLEF2026 {VERSION}</h1>
<p>Mean local validation ARI: <b>{overall_local_ari}</b></p>
<p>Submission: <code>{versioned_path}</code> and <code>{WORK_DIR / 'submission.csv'}</code></p>
<h2>Thresholds</h2><pre>{json.dumps(thresholds, indent=2)}</pre>
<h2>Local matching</h2><pre>{json.dumps(run_notes['local_matching'], indent=2)}</pre>
</body></html>
"""
(REPORT_DIR / f'run_summary_{VERSION}.html').write_text(html)
print('summary:', REPORT_DIR / f'run_summary_{VERSION}.html')


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
for candidate in [repo, *repo.parents]:
    if (candidate / "paper_submissions_manifest.csv").exists():
        repo = candidate
        break
sys.path.insert(0, str(repo / "src"))

from notebook_tools import verify_submission
verify_submission(Path.cwd() / "submission.csv", "initial_graph_baseline", repo)